# 91250 - Deep Learning 

## 0. Import and Configs

In [1]:
# File Operations
import gdown
import os
from pathlib import Path

# Math & Visualization
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau

# Helpers
from sklearn.model_selection import train_test_split
import copy
import pprint as pp

In [2]:
# Set Device to GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [3]:
# Fixing seed to reduce randomness, for better comparisons
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [4]:
# Replace these IDs with the Google Drive file IDs supplied with the project.
TRAIN_FILE_ID = "1xyggntfZ2-6BTAagxJm-tFKDXerTAs8c"
TEST_FILE_ID  = "1VozRpr3dlVpA17BL-7ALCaOXa2vCe64J"

DATA_FOLDER = "data"
TRAIN_PATH = os.path.join(DATA_FOLDER,"chess_error_detection_train.csv")
TEST_PATH  = os.path.join(DATA_FOLDER,"chess_error_detection_test.csv")

MODEL_FOLDER = "model"
os.makedirs(MODEL_FOLDER, exist_ok=True)
BEST_MODEL_PATH = os.path.join(MODEL_FOLDER,"best_model.pt")

if not os.path.exists(TRAIN_PATH):
    gdown.download(id=TRAIN_FILE_ID, output=TRAIN_PATH, quiet=False)

if not os.path.exists(TEST_PATH):
    gdown.download(id=TEST_FILE_ID, output=TEST_PATH, quiet=False)

## Data

In [5]:
train_df = pd.read_csv(TRAIN_PATH)
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["error_position"]
)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

display(train_df.head())

Train shape: (320000, 3)
Test shape:  (50000, 3)


,sequence,error_position,correct_move
91975,g3 d5 e4 c4 Nf3 Bg4 Bg2 e5 O-O Bd6 d3 Nf6 c4 d...,4,d4
252136,d4 e6 c4 d5 cxd5 Qxd5 e3 c5 Nc3 Qh5 Be2 Qg6 dx...,2,Nf6
264691,e4 e6 Qf3 c5 exd5 exd5 Nc3 Nf6 h3 Bb4 g4 O-O g...,4,d5
130957,e4 e5 Nf3 d6 Nf3 exd4 Nxd4 d5 exd5 Qxd5 Nc3 Qe...,5,d4
240371,e4 g6 d4 Bg7 Nc3 dxe4 Nge2 c6 h3 Qc7 g4 e5 Bg2...,6,d6


## Tokenization

In [6]:
unique_moves = set()
unique_moves.add("<PAD>")
unique_moves.add("<UNK>")

for game in train_df["sequence"]:
    for move in game.split():
        unique_moves.add(move)

print(len(unique_moves))

3597


In [7]:
train_vocab = unique_moves

test_moves = []

for game in test_df["sequence"]:
    test_moves.extend(game.split())

unknown_unique = set(test_moves) - train_vocab
print("Unique unseen moves:", len(unknown_unique))

unknown_count = sum(move not in train_vocab for move in test_moves)
print("Unknown count:", unknown_count)
print("Unknown rate:", unknown_count / len(test_moves))

Unique unseen moves: 99
Unknown count: 105
Unknown rate: 5.25e-05


In [8]:
vocab_to_id = {
    "<PAD>": 0,
    "<UNK>": 1
}

for move in unique_moves:
    if move not in vocab_to_id:
        vocab_to_id[move] = len(vocab_to_id)

print("Vocabulary size:", len(vocab_to_id))

Vocabulary size: 3597


In [9]:
id_to_vocab = {
    idx: move
    for move, idx in vocab_to_id.items()
}

In [10]:
def encode_sequence(sequence, vocab_to_id):
    return [
        vocab_to_id.get(move, vocab_to_id["<UNK>"])
        for move in sequence.split()
    ]

In [11]:
sequence = train_df["sequence"].iloc[0]

encoded = encode_sequence(sequence, vocab_to_id)

print(sequence)
print(encoded)
print(len(encoded))

g3 d5 e4 c4 Nf3 Bg4 Bg2 e5 O-O Bd6 d3 Nf6 c4 dxc3 Nxc3 c6 Bg5 O-O Qd2 Nbd7 Bxf6 Qxf6 Nh4 g5 Nf5 Bb4 a3 Bxc3 Qxc3 Nb6 Ne3 Bf3 Bxf3 Qxf3 Ng2 Qf6 f4 gxf4 Nxf4 Na4
[3171, 2444, 3098, 2524, 654, 3287, 2793, 3296, 2748, 2134, 2792, 3176, 2524, 920, 573, 816, 1409, 2748, 1403, 1875, 1421, 2333, 3113, 1877, 3342, 2323, 750, 3380, 165, 3049, 2836, 2603, 779, 2409, 431, 2392, 2290, 3353, 3124, 1057]
40


In [12]:
train_encoded = [
    encode_sequence(seq, vocab_to_id)
    for seq in train_df["sequence"]
]

val_encoded = [
    encode_sequence(seq, vocab_to_id)
    for seq in val_df["sequence"]
]

test_encoded = [
    encode_sequence(seq, vocab_to_id)
    for seq in test_df["sequence"]
]

In [13]:
train_labels = train_df["error_position"].to_numpy() - 1
val_labels = val_df["error_position"].to_numpy() - 1
test_labels = test_df["error_position"].to_numpy() - 1

In [14]:
x_train = torch.tensor(train_encoded, dtype=torch.long)
y_train = torch.tensor(train_labels, dtype=torch.long)

x_val = torch.tensor(val_encoded, dtype=torch.long)
y_val = torch.tensor(val_labels, dtype=torch.long)

x_test = torch.tensor(test_encoded, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

In [15]:
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

## Model Definition

In [16]:
class ChessErrorTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        nhead=4,
        num_layers=3,
        dim_feedforward=256,
        dropout=0.1,
        max_len=40
    ):
        super().__init__()

        self.d_model = d_model
        self.max_len = max_len

        # SAN move → embedding vector
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Position 0, 1, ..., 39 → embedding vector
        self.position_embedding = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Each candidate position gets one score
        self.classifier = nn.Linear(
            d_model,
            1
        )

    def forward(self, x):
        """
        x:
            [batch_size, 40]

        returns:
            [batch_size, 10]
        """

        batch_size, seq_len = x.shape

        # [40]
        positions = torch.arange(
            seq_len,
            device=x.device
        )

        # Token embeddings
        # [batch, 40, 128]
        token_embeddings = self.token_embedding(x)

        # Position embeddings
        # [40, 128]
        position_embeddings = self.position_embedding(positions)

        # Broadcasting gives:
        # [batch, 40, 128]
        x = token_embeddings + position_embeddings

        # Contextual representations
        # [batch, 40, 128]
        x = self.transformer(x)

        # We only need the first 10 positions
        # [batch, 10, 128]
        x = x[:, :10, :]

        # Score each candidate position
        # [batch, 10, 1]
        logits = self.classifier(x)

        # [batch, 10]
        logits = logits.squeeze(-1)

        return logits

In [17]:
vocab_size = len(vocab_to_id)

model = ChessErrorTransformer(
    vocab_size=vocab_size
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 863,105


## Train

In [18]:
model = model.to(DEVICE)

In [19]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",       
    factor=0.1,      
    patience=3,      
    min_lr=1e-6
)

In [20]:
PATIENCE = 5
MAX_EPOCH = 100

# Train flags
best_val_accuracy = 0.0
epochs_without_improvement = 0

In [21]:
def train(model, loader, optimizer, criterion, DEVICE):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for x, y in loader:

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(x)

        # Classification loss
        loss = criterion(logits, y)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        # Statistics
        total_loss += loss.item() * x.size(0)

        predictions = logits.argmax(dim=1)

        total_correct += (
            predictions == y
        ).sum().item()

        total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy

In [22]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)

            predictions = logits.argmax(dim=1)

            total_correct += (
                predictions == y
            ).sum().item()

            total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy

In [23]:
for epoch in range(MAX_EPOCH):
    train_loss, train_accuracy = train(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(val_accuracy)
    
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            f"New best accuracy at Epoch {epoch+1}, Val Accuracy = {val_accuracy:.4f}"
        )
    else:
        epochs_without_improvement += 1

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{MAX_EPOCH}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    # Early Stopping
    if epochs_without_improvement >= PATIENCE:
        print(
            f"Early stopping triggered at epoch {epoch+1}. "
            f"Best Val Accuracy: {best_val_accuracy:.4f}"
        )
        break


# Load best model
model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=DEVICE)
)

print(f"Best validation accuracy: {best_val_accuracy:.4f}")

New best accuracy at Epoch 1, Val Accuracy = 0.6710
Epoch [1/100] | Train Loss: 1.4779 | Train Acc: 0.4652 | Val Loss: 0.9105 | Val Acc: 0.6710 | LR: 1.00e-04
New best accuracy at Epoch 2, Val Accuracy = 0.7575
Epoch [2/100] | Train Loss: 0.8801 | Train Acc: 0.6805 | Val Loss: 0.6656 | Val Acc: 0.7575 | LR: 1.00e-04
New best accuracy at Epoch 3, Val Accuracy = 0.7974
Epoch [3/100] | Train Loss: 0.6988 | Train Acc: 0.7435 | Val Loss: 0.5523 | Val Acc: 0.7974 | LR: 1.00e-04
New best accuracy at Epoch 4, Val Accuracy = 0.8210
Epoch [4/100] | Train Loss: 0.5978 | Train Acc: 0.7794 | Val Loss: 0.4848 | Val Acc: 0.8210 | LR: 1.00e-04
New best accuracy at Epoch 5, Val Accuracy = 0.8362
Epoch [5/100] | Train Loss: 0.5311 | Train Acc: 0.8030 | Val Loss: 0.4395 | Val Acc: 0.8362 | LR: 1.00e-04
New best accuracy at Epoch 6, Val Accuracy = 0.8472
Epoch [6/100] | Train Loss: 0.4803 | Train Acc: 0.8209 | Val Loss: 0.4085 | Val Acc: 0.8472 | LR: 1.00e-04
New best accuracy at Epoch 7, Val Accuracy = 0

## Final Evaluation

In [24]:
# Evaluation mode
model.eval()

# No gradients needed
with torch.no_grad():

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion,
        DEVICE
    )

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy@1: {test_acc * 100:.2f}%")

Test Loss: 0.2615
Test Accuracy@1: 91.75%
